In [4]:
import pandas as pd
import numpy as np
import json
import re, pytz, os, requests, sys
from pathlib import Path
from datetime import datetime
import sys
sys.path.append("/workspaces/service-data")

from src.clean import clean_percentage, clean_fiscal_yr, normalize_string, standardize_column_names
from src.load import load_csv
from src.export import export_to_csv
from src.merge import merge_si, merge_ss
from src.utils import dept_list, program_list
from main import get_config

import pandas as pd
import numpy as np
import pytz
from pathlib import Path

base_dir = Path.cwd()
parent_dir = base_dir.parent
config = get_config()
# dept = dept_list(config)

# si_url = https://github.com/gcperformance/service-data/releases/latest/download/si.csv
si_input_path = parent_dir / 'inputs' / 'si_2024.csv'
si_input = pd.read_csv(si_input_path, keep_default_na=False)

si_path = parent_dir / 'outputs' / 'si.csv'
si = pd.read_csv(si_path, keep_default_na=False, na_values='', delimiter=';')

si_all_path = parent_dir / 'outputs' / 'utils'/'si_all.csv'
si_all = pd.read_csv(si_all_path, keep_default_na=False, na_values='', delimiter=';')

# sid_registry = load_csv(sid_registry_path, config)

In [8]:
# def publish_release(config):
'''
Return True if any configured local table differs from its live version
'''
# Compare each configured local file with its live counterpart.
# Pull relevant info from config file, convert to DataFrame for ease of use
local_and_live_paths = config['local_and_live_paths']
local_and_live_paths = pd.DataFrame(local_and_live_paths, columns=['table', 'local_path', 'live_url'])
publish_flag = False

# Iterate over the set of tables
for table in local_and_live_paths['table']:
    # Read the local file
    local_path = local_and_live_paths.loc[local_and_live_paths['table']==table, 'local_path'].item()
    local = pd.read_csv(local_path, dtype=str, keep_default_na=False, delimiter=';', engine='python', skipfooter=2)

    # Read the remote file
    live_url =  local_and_live_paths.loc[local_and_live_paths['table']==table, 'live_url'].item()
    live = pd.read_csv(live_url, dtype=str, keep_default_na=False, delimiter=';', engine='python', skipfooter=2)

    # Create a boolean mask of which cells differ from one DF to the other 
    # ().ne() produces True if the cells differ; ne = not equal)
    # then melt into a single column
    check = live.ne(local).melt()

    # Filter the single column for entries that are True (i.e. differences)
    differences = check[check['value']]

    # Publish flag is true when there are differences to report
    # if the length of the differences column is greater than 0, then 
    publish_flag = differences.shape[0] > 0

    # Stop once a difference is found
    if publish_flag:
        break


print(publish_flag)

True
